# Audit Visualization Notebook

This notebook visualizes outputs from:
- `outputs/coverage_audits`
- `outputs/annotation_audits`
- `outputs/disparity_audits`

It is intended to run after audit stages 00, 01, and 02.

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('ggplot')

In [ ]:
WORKDIR = Path.cwd()
if not (WORKDIR / 'outputs').exists():
    WORKDIR = WORKDIR.parent

coverage_dir = WORKDIR / 'outputs' / 'coverage_audits'
annotation_dir = WORKDIR / 'outputs' / 'annotation_audits'
disparity_dir = WORKDIR / 'outputs' / 'disparity_audits'
viz_output_dir = WORKDIR / 'outputs' / 'audit_visualizations'
viz_output_dir.mkdir(parents=True, exist_ok=True)

def read_tsv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path}')
    return pd.read_csv(path, sep='\t')

audit_metrics = read_tsv(coverage_dir / 'audit_metrics.tsv')
audit_detailed = read_tsv(coverage_dir / 'audit_detailed.tsv')
annotation_quality = read_tsv(annotation_dir / 'annotation_quality.tsv')
case_breakdown = read_tsv(annotation_dir / 'case_breakdown.tsv')
form_labeling = read_tsv(annotation_dir / 'form_labeling_detail.tsv')
coverage_disparity = read_tsv(disparity_dir / 'coverage_disparity.tsv')
annotation_disparity = read_tsv(disparity_dir / 'annotation_disparity.tsv')
cross_level_consistency = read_tsv(disparity_dir / 'cross_level_consistency.tsv')

coarse_levels = sorted(audit_metrics['taxonomy_level'].dropna().astype(str).unique().tolist())

def make_facet_axes(levels: list[str], ncols: int = 3, panel_w: float = 5.0, panel_h: float = 4.0):
    n = len(levels)
    ncols = max(1, min(ncols, n))
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel_w * ncols, panel_h * nrows))
    axes = np.atleast_1d(axes).reshape(nrows, ncols).flatten()
    for idx in range(len(axes)):
        if idx >= n:
            axes[idx].axis('off')
    return fig, axes

print('Loaded datasets:')
for name, df in [
    ('audit_metrics', audit_metrics),
    ('audit_detailed', audit_detailed),
    ('annotation_quality', annotation_quality),
    ('case_breakdown', case_breakdown),
    ('form_labeling', form_labeling),
    ('coverage_disparity', coverage_disparity),
    ('annotation_disparity', annotation_disparity),
    ('cross_level_consistency', cross_level_consistency),
]:
    print(f'  - {name}: {len(df)} rows')

print('Facet levels:', coarse_levels)

## Coverage Visualizations

In [ ]:
fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=5.6, panel_h=4.6)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    plot_df = (
        audit_metrics[audit_metrics['taxonomy_level'] == level]
        .sort_values('token_frequency', ascending=False)
        .head(10)
        .copy()
    )

    if plot_df.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(level)
        ax.set_axis_off()
        continue

    plot_df = plot_df.sort_values('token_frequency', ascending=True)
    ax.barh(plot_df['target'], plot_df['token_frequency'], color='#2f6db5')
    ax.set_title(level)
    ax.set_xlabel('Token frequency')
    ax.set_ylabel('Target')

fig.suptitle('Top token-frequency targets per coarse label', y=1.01)
plt.tight_layout()
fig.savefig(viz_output_dir / 'coverage_top20_token_frequency.png', dpi=180)
plt.show()

In [ ]:
norm = mpl.colors.Normalize(
    vmin=max(float(audit_metrics['token_frequency'].min()), 1.0),
    vmax=max(float(audit_metrics['token_frequency'].max()), 1.0),
)

fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=5.6, panel_h=4.6)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    level_df = audit_metrics[audit_metrics['taxonomy_level'] == level].copy()

    if level_df.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(level)
        ax.set_axis_off()
        continue

    sizes = np.clip(level_df['token_frequency'].to_numpy(), 1, None)
    sizes = 30 + 150 * (sizes / sizes.max())

    ax.scatter(
        level_df['presence_rate'],
        level_df['type_coverage'],
        s=sizes,
        c=level_df['token_frequency'],
        cmap='viridis',
        norm=norm,
        alpha=0.8,
        edgecolor='black',
        linewidth=0.3,
    )
    ax.set_title(level)
    ax.set_xlabel('Presence rate')
    ax.set_ylabel('Type coverage')
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)

sm = mpl.cm.ScalarMappable(norm=norm, cmap='viridis')
sm.set_array([])
fig.colorbar(sm, ax=axes[: len(coarse_levels)], shrink=0.72, label='Token frequency')
fig.suptitle('Coverage quality by coarse label', y=1.01)
plt.tight_layout()
fig.savefig(viz_output_dir / 'coverage_presence_vs_type.png', dpi=180)
plt.show()

## Annotation Quality Visualizations

In [ ]:
fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=6.2, panel_h=4.8)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    annot_plot = (
        annotation_quality[annotation_quality['taxonomy_level'] == level]
        .sort_values('total_matches', ascending=False)
        .head(8)
        .copy()
    )

    if annot_plot.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(level)
        ax.set_axis_off()
        continue

    x = np.arange(len(annot_plot))
    width = 0.4
    ax.bar(
        x - width / 2,
        annot_plot['correct_labeling_rate'],
        width=width,
        label='correct',
        color='#1b9e77',
    )
    ax.bar(
        x + width / 2,
        annot_plot['annotator_failure_ratio'],
        width=width,
        label='failure',
        color='#d95f02',
    )
    ax.set_xticks(x)
    ax.set_xticklabels(annot_plot['target'], rotation=50, ha='right')
    ax.set_ylim(0, 1)
    ax.set_title(level)
    ax.set_ylabel('Rate')
    ax.legend(frameon=False, fontsize=8)

fig.suptitle('Annotation quality rates by coarse label', y=1.01)
plt.tight_layout()
fig.savefig(viz_output_dir / 'annotation_quality_top20_rates.png', dpi=180)
plt.show()

In [ ]:
fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=5.8, panel_h=4.8)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    case_plot = (
        annotation_quality[annotation_quality['taxonomy_level'] == level]
        .sort_values('total_matches', ascending=False)
        .head(8)
        .sort_values('total_matches', ascending=True)
        .copy()
    )

    if case_plot.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(level)
        ax.set_axis_off()
        continue

    ax.barh(case_plot['target'], case_plot['case_a_present_hateful'], color='#1b9e77', label='Case A')
    ax.barh(
        case_plot['target'],
        case_plot['case_b_present_nonhateful'],
        left=case_plot['case_a_present_hateful'],
        color='#d95f02',
        label='Case B',
    )
    ax.set_title(level)
    ax.set_xlabel('Matched count')
    ax.set_ylabel('Target')
    ax.legend(frameon=False, fontsize=8)

fig.suptitle('Case A/B composition by coarse label', y=1.01)
plt.tight_layout()
fig.savefig(viz_output_dir / 'annotation_case_ab_top15.png', dpi=180)
plt.show()

## Disparity Visualizations

In [ ]:
fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=5.8, panel_h=4.4)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    level_df = coverage_disparity[coverage_disparity['taxonomy_level'] == level]

    if level_df.empty:
        ax.text(0.5, 0.5, 'No pairwise rows', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(level)
        ax.set_axis_off()
        continue

    ax.hist(level_df['presence_rate_di_ratio'], bins=12, color='#4c78a8', alpha=0.65, label='presence DI')
    ax.hist(level_df['type_coverage_di_ratio'], bins=12, color='#72b7b2', alpha=0.65, label='type DI')
    ax.axvline(0.8, color='red', linestyle='--', linewidth=1.3)
    ax.set_title(level)
    ax.set_xlabel('DI ratio')
    ax.set_ylabel('Pair count')
    ax.legend(frameon=False, fontsize=8)

fig.suptitle('DI-ratio distributions by coarse label', y=1.01)
plt.tight_layout()
fig.savefig(viz_output_dir / 'disparity_di_ratio_histograms.png', dpi=180)
plt.show()

In [ ]:
fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=6.4, panel_h=5.0)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    worst_di = coverage_disparity[coverage_disparity['taxonomy_level'] == level].copy()

    if worst_di.empty:
        ax.text(0.5, 0.5, 'No pairwise rows', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(level)
        ax.set_axis_off()
        continue

    worst_di['pair'] = worst_di['target_a'] + ' vs ' + worst_di['target_b']
    worst_di['worst_di_ratio'] = worst_di[['presence_rate_di_ratio', 'type_coverage_di_ratio']].min(axis=1)
    worst_di = worst_di.sort_values('worst_di_ratio', ascending=True).head(8)

    ax.barh(worst_di['pair'], worst_di['worst_di_ratio'], color='#b279a2')
    ax.axvline(0.8, color='red', linestyle='--', linewidth=1.3)
    ax.set_title(level)
    ax.set_xlabel('Worst DI ratio')
    ax.set_xlim(0, 1.05)

fig.suptitle('Lowest DI-ratio comparisons by coarse label', y=1.01)
plt.tight_layout()
fig.savefig(viz_output_dir / 'disparity_worst_di_top20.png', dpi=180)
plt.show()

In [ ]:
fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=6.4, panel_h=5.0)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    ann_gap = annotation_disparity[annotation_disparity['taxonomy_level'] == level].copy()

    if ann_gap.empty:
        ax.text(0.5, 0.5, 'No pairwise rows', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(level)
        ax.set_axis_off()
        continue

    ann_gap['pair'] = ann_gap['target_a'] + ' vs ' + ann_gap['target_b']
    ann_gap = ann_gap.sort_values('labeling_rate_gap_abs', ascending=False).head(8).sort_values('labeling_rate_gap_abs')

    ax.barh(ann_gap['pair'], ann_gap['labeling_rate_gap_abs'], color='#e45756')
    ax.set_title(level)
    ax.set_xlabel('Absolute correct-labeling-rate gap')

fig.suptitle('Annotation quality gaps by coarse label', y=1.01)
plt.tight_layout()
fig.savefig(viz_output_dir / 'annotation_disparity_label_gap_top20.png', dpi=180)
plt.show()

In [ ]:
if cross_level_consistency.empty:
    print('cross_level_consistency is empty for this run.')
else:
    fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=6.8, panel_h=4.8)

    for idx, level in enumerate(coarse_levels):
        ax = axes[idx]
        level_df = cross_level_consistency[cross_level_consistency['level_from'] == level].copy()

        if level_df.empty:
            ax.text(0.5, 0.5, 'No transitions from this level', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(level)
            ax.set_axis_off()
            continue

        labels = level_df['target'] + ' -> ' + level_df['level_to']
        y = np.arange(len(level_df))

        ax.barh(y - 0.2, level_df['presence_rate_delta'], height=0.38, color='#4c78a8', label='presence delta')
        ax.barh(y + 0.2, level_df['correct_labeling_rate_delta'], height=0.38, color='#54a24b', label='labeling delta')
        ax.set_yticks(y)
        ax.set_yticklabels(labels)
        ax.axvline(0, color='black', linewidth=1)
        ax.set_title(level)
        ax.set_xlabel('Delta (to - from)')
        ax.legend(frameon=False, fontsize=8)

    fig.suptitle('Cross-level deltas faceted by source coarse label', y=1.01)
    plt.tight_layout()
    fig.savefig(viz_output_dir / 'cross_level_deltas.png', dpi=180)
    plt.show()

In [ ]:
print(f'Visualization images written to: {viz_output_dir}')